In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
df_train = pd.read_csv('../input/titanic/train.csv')
df_test = pd.read_csv('../input/titanic/test.csv')

# Cleaning Data

### according to this [Notebook](https://www.kaggle.com/mohsenpg/logisticregression-on-titanic) i'll do cleaning data 

In [3]:
def missing_percent(df):
    nan_percent = 100* (df.isnull().sum() / len(df))
    nan_percent = nan_percent[nan_percent>0].sort_values()
    return nan_percent

df_train = df_train.dropna(axis=0,subset=['Embarked'])
df_train['Age'] = df_train.groupby('Sex')['Age'].transform(lambda x: x.fillna(x.mean()))
df_train = df_train.drop(['Cabin'],axis=1)

df_Sex_train = pd.get_dummies(df_train['Sex'])
df_train = df_train.drop('Sex',axis=1)
df_train = pd.concat([df_train,df_Sex_train], axis=1)


df_test = df_test.dropna(axis=0,subset=['Embarked','Fare'])

df_test['Age'] = df_test.groupby('Sex')['Age'].transform(lambda x: x.fillna(x.mean()))
df_test = df_test.drop(['Cabin'],axis=1)

df_Sex_test = pd.get_dummies(df_test['Sex'])
df_test = df_test.drop('Sex',axis=1)
df_test = pd.concat([df_test,df_Sex_test], axis=1)

## this time we want to see our clusters

In [4]:
df_train['Survived'].value_counts()

0    549
1    340
Name: Survived, dtype: int64

#### we have two groups:
#### - first is 0 that means capacity of 549 haven't survived
#### - second is 1 that means capacity of 549 have survived

# Determine The Features and Target Value(Label)

In [5]:
X = df_train.drop(['Survived','Name','Ticket','Embarked'],axis=1)
y = df_train['Survived']

# Spliting The Data to Train and Test

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

# Feature Scaling: Scaling The Features

In [7]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [8]:
scaler.fit(X_train)
scaled_X_train = scaler.transform(X_train)
scaled_X_test  = scaler.transform(X_test)

# Train The Model

In [9]:
from sklearn.svm import SVC
model = SVC()
model.fit(X_train, y_train)

SVC()

In [10]:
y_pred= model.predict(X_test)

# The Accuracy

In [11]:
from sklearn.metrics import classification_report, confusion_matrix,plot_roc_curve

In [12]:
confusion_matrix(y_test, y_pred)

array([[163,   4],
       [ 91,   9]])

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.98      0.77       167
           1       0.69      0.09      0.16       100

    accuracy                           0.64       267
   macro avg       0.67      0.53      0.47       267
weighted avg       0.66      0.64      0.54       267



In [14]:
from sklearn.model_selection import GridSearchCV

In [15]:
svm = SVC()
param_grid = {'C':[0.01,0.1,1,10,100,1000],'gamma':[1,0.1,0.01,0.001,0.0001]}
grid = GridSearchCV(svm, param_grid, cv = 5)

In [16]:
grid.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100, 1000],
                         'gamma': [1, 0.1, 0.01, 0.001, 0.0001]})

In [17]:
grid.best_estimator_

SVC(C=1000, gamma=0.0001)

In [18]:
grid.best_params_

{'C': 1000, 'gamma': 0.0001}

In [19]:
y_pred_grid = grid.predict(X_test)

In [20]:
confusion_matrix(y_test, y_pred_grid)

array([[134,  33],
       [ 30,  70]])

In [21]:
print(classification_report(y_test, y_pred_grid))

              precision    recall  f1-score   support

           0       0.82      0.80      0.81       167
           1       0.68      0.70      0.69       100

    accuracy                           0.76       267
   macro avg       0.75      0.75      0.75       267
weighted avg       0.77      0.76      0.76       267



# Predect Test Data

#### Now this time we must predict the test data set
#### before we was training and test the train data set
#### to be sure about the accuracy of our model
#### now we must test the test dataset.

In [22]:
X_test  = df_test.drop(['Name','Ticket','Embarked'],axis=1)
scaled_X_test = scaler.transform(X_test)
y_pred_grid = grid.predict(scaled_X_test)

In [23]:
y_pred_grid = pd.Series(y_pred_grid)
df_test = pd.concat([df_test,y_pred_grid], axis=0)
df_test = df_test.rename(columns={0: "Survived"})

In [24]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 834 entries, 0 to 416
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     417 non-null    float64
 1   Age          417 non-null    float64
 2   Embarked     417 non-null    object 
 3   Fare         417 non-null    float64
 4   Name         417 non-null    object 
 5   Parch        417 non-null    float64
 6   PassengerId  417 non-null    float64
 7   Pclass       417 non-null    float64
 8   SibSp        417 non-null    float64
 9   Ticket       417 non-null    object 
 10  female       417 non-null    float64
 11  male         417 non-null    float64
dtypes: float64(9), object(3)
memory usage: 84.7+ KB


In [25]:
df_test.Survived = df_test.Survived.fillna(0)
df_test.Survived = df_test.Survived.astype(int)

In [26]:
df_test

,Survived,Age,Embarked,Fare,Name,Parch,PassengerId,Pclass,SibSp,Ticket,female,male
0,0,34.5,Q,7.8292,"Kelly, Mr. James",0.0,892.0,3.0,0.0,330911,0.0,1.0
1,0,47.0,S,7.0000,"Wilkes, Mrs. James (Ellen Needs)",0.0,893.0,3.0,1.0,363272,1.0,0.0
2,0,62.0,Q,9.6875,"Myles, Mr. Thomas Francis",0.0,894.0,2.0,0.0,240276,0.0,1.0
3,0,27.0,S,8.6625,"Wirz, Mr. Albert",0.0,895.0,3.0,0.0,315154,0.0,1.0
4,0,22.0,S,12.2875,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",1.0,896.0,3.0,1.0,3101298,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
412,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
413,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
414,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
415,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
